# 🎵 Treinamento e Preparação de Dados Musical
### Projeto Integrador VI — Sistemas de Informação (PUC-Campinas, 2026) — Grupo B6

Notebook autocontido para o Google Colab (GPU T4). Faz o pipeline de dados (scripts 01 a 05 do repositório), treina o **Transformer** e a **LSTM** e gera estatísticas de treinamento e de avaliação.

Tudo que precisa persistir fica no Google Drive, em `Meus Drives/Colab Notebooks/SI-PI6-2026-G06/`:
- `dados_processados/` (sequências, vocabulário, divisão treino/validação/teste)
- `modelos/` (checkpoints `modelo_transformer.pt` e `modelo_lstm.pt`, com vocabulário)
- `relatorios/` (histórico por época, resultados de teste, gráfico e resumo do pipeline)

Ordem de execução: célula 1 em diante. Se a sessão cair, rode novamente da célula 6 em diante; os dados e checkpoints já gravados no Drive são reaproveitados.

--- 
## 1. Verificação da GPU T4

In [ ]:
!nvidia-smi

--- 
## 2. Instalação das Dependências

In [ ]:
!pip install -q torch pretty_midi tqdm numpy scikit-learn

--- 
## 3. Montar o Google Drive
O `lmd_full.tar.gz` deve estar em `Meus Drives/Colab Notebooks/SI-PI6-2026-G06/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/Colab Notebooks/SI-PI6-2026-G06')
TAR_DRIVE = DRIVE / 'lmd_full.tar.gz'
assert TAR_DRIVE.exists(), f'Arquivo não encontrado no Drive: {TAR_DRIVE}'
print('Drive OK:', DRIVE)

--- 
## 4. Configuração (equivalente ao `configuracao/pipeline.yaml`)
Valores idênticos ao YAML e às constantes dos scripts 01 a 05. Se o YAML mudar, altere também esta célula.

In [ ]:
import json, random
import numpy as np

# piloto
PILOTO_ATIVO = True
PILOTO_QUANTIDADE = 100000

# filtros (pipeline.yaml > filtros)
NOTAS_MINIMAS = 32
NOTAS_MAXIMAS = 2000
DURACAO_MINIMA_SEG = 15.0
DURACAO_MAXIMA_SEG = 600.0
PITCH_MINIMO = 36
PITCH_MAXIMO = 96

# normalização (pipeline.yaml > normalizacao)
GRADES_RITMICAS = [0.125, 0.25, 0.5, 1.0, 2.0]

# divisão (pipeline.yaml > divisao)
PROPORCAO_TREINO = 0.80
PROPORCAO_VALIDACAO = 0.10
SEMENTE = 42

# constantes dos scripts
TAMANHO_MINIMO_ARQUIVO_BYTES = 200   # script 02
LIMITE_SIMULTANEO_SEG = 0.05         # script 03 (50 ms)
LIMITE_PITCH_FORA_PCT = 0.05         # script 03
LIMITE_MELODIA_PLANA = 0.80          # script 03
LIMITE_REST_SEG = 0.01               # script 04
EXTENSOES = ('.mid', '.midi')
PESOS = {'notas': 0.3, 'registro': 0.2, 'variacao': 0.3, 'polifonia': 0.1, 'pausas': 0.1}  # script 03

print(f'Piloto: {PILOTO_QUANTIDADE:,} arquivos (ativo={PILOTO_ATIVO})')

--- 
## 5. Descompactar o `lmd_full.tar.gz` no SSD do Colab
A extração vai para o SSD do Colab (rápido). Só é feita se ainda não existir.

In [ ]:
import subprocess

PASTA_BRUTOS = Path('/content/arquivos_brutos')
PASTA_BRUTOS.mkdir(parents=True, exist_ok=True)

if not any(PASTA_BRUTOS.rglob('*.mid*')):
    print('Descompactando arquivos brutos...')
    subprocess.run(['tar', '-xzf', str(TAR_DRIVE), '-C', str(PASTA_BRUTOS)], check=True)
    print('Descompactação concluída.')
else:
    print('Arquivos brutos já disponíveis no SSD do Colab.')

--- 
## 6. Pipeline de Dados (scripts 01 a 05)
Etapas: **01** inventário (ordenação e piloto) → **02** validação (corrompido, sem notas, duração, tamanho) → **03** seleção de faixa por pontuação, monofonia e filtros finais (inclui melodia plana) → **04** tokenização → **05** divisão por `train_test_split` com semente fixa.

O piloto processa **100.000 arquivos MIDI** de entrada. Nem todos são aprovados: o número de sequências aprovadas e os descartes por motivo são impressos e gravados em `relatorios/resumo_pipeline.json`.

In [ ]:
import pretty_midi
from sklearn.model_selection import train_test_split
from tqdm import tqdm

PASTA_DADOS = DRIVE / 'dados_processados'
PASTA_RELATORIOS = DRIVE / 'relatorios'
pasta_seq = PASTA_DADOS / 'sequencias'
pastas = {
    'treino': PASTA_DADOS / 'treino',
    'validacao': PASTA_DADOS / 'validacao',
    'teste': PASTA_DADOS / 'teste',
}
for p in [pasta_seq, PASTA_RELATORIOS, *pastas.values()]:
    p.mkdir(parents=True, exist_ok=True)

# Etapa 01: lista ordenada de todos os MIDIs e depois o piloto (mesmo critério do script 01)
todos = sorted(p for p in PASTA_BRUTOS.rglob('*') if p.suffix.lower() in EXTENSOES)
print(f'MIDIs brutos localizados: {len(todos):,}')
if PILOTO_ATIVO:
    todos = todos[:PILOTO_QUANTIDADE]
print(f'Arquivos de entrada do piloto: {len(todos):,}')

def quantizar(duracao_beats):
    grades = np.array(GRADES_RITMICAS)
    return f'{grades[int(np.argmin(np.abs(grades - duracao_beats)))]:.2f}'

def bpm_do_midi(pm):
    tempos = pm.get_tempo_changes()
    return float(tempos[1][0]) if len(tempos[1]) > 0 else 120.0

def pontuacao_faixa(inst, duracao_total_midi):
    notas = inst.notes
    num = len(notas)
    pitches = [n.pitch for n in notas]
    pitch_medio = np.mean(pitches)
    variacao = np.std(pitches)
    ordenadas = sorted(notas, key=lambda n: n.start)
    sobrepostos = sum(1 for i in range(1, num) if ordenadas[i].start < ordenadas[i - 1].end - 0.001)
    pct_polifonia = sobrepostos / num
    duracao_total = ordenadas[-1].end - ordenadas[0].start
    tempo_com_nota = sum(n.end - n.start for n in ordenadas)
    pct_pausas = max(0.0, 1.0 - tempo_com_nota / duracao_total) if duracao_total > 0 else 1.0
    norm_notas = min(num / 500.0, 1.0)
    norm_registro = min(max((pitch_medio - 36) / 60.0, 0.0), 1.0)
    norm_var = min(variacao / 12.0, 1.0)
    return (PESOS['notas'] * norm_notas + PESOS['registro'] * norm_registro
            + PESOS['variacao'] * norm_var - PESOS['polifonia'] * pct_polifonia
            - PESOS['pausas'] * pct_pausas)

def selecionar_faixa(pm):
    melhor, melhor_score = None, -999.0
    duracao = pm.get_end_time()
    for inst in pm.instruments:
        if inst.is_drum or len(inst.notes) < NOTAS_MINIMAS:
            continue
        if duracao < DURACAO_MINIMA_SEG:
            continue
        score = pontuacao_faixa(inst, duracao)
        if score > melhor_score:
            melhor, melhor_score = inst, score
    return melhor

def monofonia(notas):
    ordenadas = sorted(notas, key=lambda n: n.start)
    resultado, grupo = [], [ordenadas[0]]
    for nota in ordenadas[1:]:
        if nota.start - grupo[-1].start < LIMITE_SIMULTANEO_SEG:
            grupo.append(nota)
        else:
            resultado.append(max(grupo, key=lambda n: n.pitch))
            grupo = [nota]
    resultado.append(max(grupo, key=lambda n: n.pitch))
    return resultado

def filtros_finais(notas):
    fora = [n for n in notas if not (PITCH_MINIMO <= n.pitch <= PITCH_MAXIMO)]
    if len(fora) / len(notas) > LIMITE_PITCH_FORA_PCT:
        notas = [n for n in notas if PITCH_MINIMO <= n.pitch <= PITCH_MAXIMO]
    if len(notas) < NOTAS_MINIMAS:
        return None, 'NOTAS_INSUFICIENTES'
    if len(notas) > NOTAS_MAXIMAS:
        return None, 'NOTAS_EXCESSIVAS'
    contagem = {}
    for n in notas:
        contagem[n.pitch] = contagem.get(n.pitch, 0) + 1
    if sum(sorted(contagem.values(), reverse=True)[:2]) / len(notas) > LIMITE_MELODIA_PLANA:
        return None, 'MELODIA_PLANA'
    return notas, None

def tokenizar(notas, bpm):
    spb = bpm / 60.0
    tokens = []
    for i, n in enumerate(notas):
        if i > 0:
            gap = n.start - notas[i - 1].end
            if gap > LIMITE_REST_SEG:
                tokens.append(f'REST_{quantizar(gap * spb)}')
        tokens += [f'NOTE_{n.pitch}', f'DURATION_{quantizar((n.end - n.start) * spb)}']
    return tokens

descartes = {'arquivo_minimo': 0, 'corrompido': 0, 'sem_notas': 0, 'curto_demais': 0,
             'longo_demais': 0, 'sem_faixa': 0, 'notas_insuficientes': 0,
             'notas_excessivas': 0, 'melodia_plana': 0}
sequencias = []  # (md5, bpm, duracao, tokens)

for caminho in tqdm(todos, desc='Processando MIDIs'):
    if caminho.stat().st_size < TAMANHO_MINIMO_ARQUIVO_BYTES:
        descartes['arquivo_minimo'] += 1
        continue
    try:
        pm = pretty_midi.PrettyMIDI(str(caminho))
    except Exception:
        descartes['corrompido'] += 1
        continue
    duracao = pm.get_end_time()
    if sum(len(i.notes) for i in pm.instruments) == 0:
        descartes['sem_notas'] += 1
        continue
    if duracao < DURACAO_MINIMA_SEG:
        descartes['curto_demais'] += 1
        continue
    if duracao > DURACAO_MAXIMA_SEG:
        descartes['longo_demais'] += 1
        continue
    faixa = selecionar_faixa(pm)
    if faixa is None:
        descartes['sem_faixa'] += 1
        continue
    notas, motivo = filtros_finais(monofonia(faixa.notes))
    if motivo:
        descartes[motivo.lower()] += 1
        continue
    tokens = tokenizar(notas, bpm_do_midi(pm))
    if not tokens:
        descartes['sem_faixa'] += 1
        continue
    sequencias.append((caminho.stem, round(bpm_do_midi(pm), 2), round(duracao, 3), tokens))

print()
print(f'Sequências aprovadas: {len(sequencias):,} de {len(todos):,} arquivos de entrada')
print(f'Descartes: {descartes}')

# Etapa 04: vocabulário em ordem de aparição, especiais no fim (mesmo critério do script 04)
vocabulario = {}
for _, _, _, tokens in sequencias:
    for tok in tokens:
        if tok not in vocabulario:
            vocabulario[tok] = len(vocabulario)
for especial in ['<PAD>', '<START>', '<END>']:
    if especial not in vocabulario:
        vocabulario[especial] = len(vocabulario)
vocab_json = {
    'token_para_id': vocabulario,
    'id_para_token': {str(v): k for k, v in vocabulario.items()},
    'tamanho_vocabulario': len(vocabulario),
    'tokens_especiais': {'PAD': '<PAD>', 'START': '<START>', 'END': '<END>'},
}
(pasta_seq / 'vocabulario.json').write_text(json.dumps(vocab_json, ensure_ascii=False, indent=2), encoding='utf-8')

# Etapa 05: divisão em duas chamadas de train_test_split com a mesma semente (script 05)
md5s = [s[0] for s in sequencias]
por_md5 = {s[0]: s for s in sequencias}
md5_treino, md5_resto = train_test_split(md5s, test_size=1.0 - PROPORCAO_TREINO, random_state=SEMENTE, shuffle=True)
pct_val_do_resto = PROPORCAO_VALIDACAO / (1.0 - PROPORCAO_TREINO)
md5_val, md5_teste = train_test_split(md5_resto, test_size=1.0 - pct_val_do_resto, random_state=SEMENTE, shuffle=True)

for nome, lista in {'treino': md5_treino, 'validacao': md5_val, 'teste': md5_teste}.items():
    for md5 in lista:
        _, bpm, duracao, tokens = por_md5[md5]
        (pastas[nome] / f'{md5}.json').write_text(json.dumps({
            'md5': md5, 'bpm': bpm, 'duracao_seg': duracao, 'num_tokens': len(tokens), 'tokens': tokens,
        }, ensure_ascii=False, separators=(',', ':')), encoding='utf-8')
    (pastas[nome] / 'vocabulario.json').write_text(json.dumps(vocab_json, ensure_ascii=False, indent=2), encoding='utf-8')

resumo = {
    'piloto_quantidade': PILOTO_QUANTIDADE,
    'arquivos_entrada': len(todos),
    'sequencias_aprovadas': len(sequencias),
    'descartes': descartes,
    'divisao': {'treino': len(md5_treino), 'validacao': len(md5_val), 'teste': len(md5_teste)},
    'vocabulario_tamanho': len(vocabulario),
}
(PASTA_RELATORIOS / 'resumo_pipeline.json').write_text(json.dumps(resumo, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Treino: {len(md5_treino):,} | Validação: {len(md5_val):,} | Teste: {len(md5_teste):,}')
print(f'Vocabulário: {len(vocabulario)} tokens')

--- 
## 7. Funções de Treino, Avaliação e Geração (Transformer e LSTM)
Define o carregamento das janelas, as arquiteturas (iguais às de `src/modelos/`), as métricas e as rotinas de geração.

Métricas calculadas em cada conjunto (treino, validação e teste), sobre tokens que não são `<PAD>`:
- **Loss** (cross-entropy) e **perplexidade** (exp(loss))
- **Acurácia** (token previsto = token real) e **Top-5**
- **Precisão, Recall e F1 macro** (média sobre as classes presentes no alvo)
- **AUC-ROC** one-vs-rest macro (amostra de até 200 mil tokens, só no teste)

Geração: 50 melodias por modelo, com as mesmas regras de amostragem de `src/modelos/transformer_model.py` (gramática NOTE→DURATION→NOTE/REST, penalidade de repetição, top-k/top-p). A LSTM não tem gramática no repositório, então é gerada com top-k simples, como em `src/modelos/lstm_model.py`. As estatísticas de melodia comparam as gerações com as melodias reais do teste.

In [ ]:
import csv, math, time
from collections import Counter
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)

vocab_json = json.loads((pasta_seq / 'vocabulario.json').read_text(encoding='utf-8'))
token_para_id = vocab_json['token_para_id']
id_para_token = vocab_json['id_para_token']
VOCAB_SIZE = len(token_para_id)
PAD_ID = token_para_id['<PAD>']
START_ID = token_para_id['<START>']
END_ID = token_para_id['<END>']
ID_DUR_MIN = token_para_id.get('DURATION_0.12')

# Hiperparâmetros (iguais a scripts/treinar_neural.py)
SEQ_LEN = 128
BATCH_SIZE = 32
LR = 1e-3
D_MODEL, NHEAD, NUM_LAYERS, DIM_FF, MAX_SEQ_LEN = 256, 4, 4, 512, 1024
DROPOUT_TRANSFORMER, DROPOUT_LSTM = 0.1, 0.2
EPOCAS_TRANSFORMER, EPOCAS_LSTM = 15, 20
N_MELODIAS_GERADAS = 50
TOKENS_POR_MELODIA = 256

PASTA_MODELOS = DRIVE / 'modelos'
PASTA_MODELOS.mkdir(parents=True, exist_ok=True)

def janelas_de(pasta):
    janelas = []
    for arq in sorted(pasta.glob('*.json')):
        if arq.stem == 'vocabulario':
            continue
        tokens = json.loads(arq.read_text(encoding='utf-8'))['tokens']
        if len(tokens) < 8:
            continue
        ids = [START_ID] + [token_para_id.get(t, PAD_ID) for t in tokens] + [END_ID]
        for i in range(0, max(1, len(ids) - SEQ_LEN), max(1, SEQ_LEN // 2)):
            janela = ids[i : i + SEQ_LEN + 1]
            if len(janela) > 4:
                janelas.append(np.array(janela + [PAD_ID] * (SEQ_LEN + 1 - len(janela)), dtype=np.int16))
    return np.stack(janelas) if janelas else np.zeros((0, SEQ_LEN + 1), dtype=np.int16)

print('Carregando janelas do Drive...')
dados_treino = janelas_de(pastas['treino'])
dados_val = janelas_de(pastas['validacao'])
dados_teste = janelas_de(pastas['teste'])
print(f'Janelas | treino: {len(dados_treino):,} | validação: {len(dados_val):,} | teste: {len(dados_teste):,}')
if len(dados_treino) == 0:
    raise ValueError('Sem dados de treino. Execute a célula 6 primeiro.')

class JanelasDataset(Dataset):
    def __init__(self, dados):
        self.dados = dados

    def __len__(self):
        return len(self.dados)

    def __getitem__(self, i):
        w = self.dados[i].astype(np.int64)
        return torch.from_numpy(w[:-1]), torch.from_numpy(w[1:])

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]

class MusicTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, dim_feedforward, dropout, max_seq_len):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_seq_len)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
                                           dropout=dropout, batch_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        mask = torch.triu(torch.full((seq_len, seq_len), float('-inf'), device=x.device), diagonal=1)
        embeds = self.embedding(x) * math.sqrt(self.d_model)
        out = self.transformer(self.pos_encoder(embeds), mask=mask, is_causal=True)
        return self.fc_out(out)

class MelodyLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            dropout=dropout if num_layers > 1 else 0.0, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        out, hidden = self.lstm(self.embedding(x), hidden)
        return self.fc(self.dropout(out)), hidden

def criar_modelo(nome):
    if nome == 'transformer':
        return MusicTransformer(VOCAB_SIZE, D_MODEL, NHEAD, NUM_LAYERS, DIM_FF, DROPOUT_TRANSFORMER, MAX_SEQ_LEN).to(device)
    return MelodyLSTM(VOCAB_SIZE, D_MODEL, DIM_FF, NUM_LAYERS, DROPOUT_LSTM).to(device)

def logits_de(modelo, x):
    saida = modelo(x)
    return saida[0] if isinstance(saida, tuple) else saida

def auc_ovr(probs, y):
    aucs = []
    for c in np.unique(y):
        positivos = y == c
        if positivos.sum() == 0 or positivos.sum() == len(y):
            continue
        aucs.append(roc_auc_score(positivos, probs[:, c]))
    return float(np.mean(aucs)) if aucs else float('nan')

def avaliar(modelo, dl, calcular_auc=False, max_auc=200000):
    modelo.eval()
    criterio_soma = nn.CrossEntropyLoss(ignore_index=PAD_ID, reduction='sum')
    perda, total, acertos, top5 = 0.0, 0, 0, 0
    tp = torch.zeros(VOCAB_SIZE)
    pred_c = torch.zeros(VOCAB_SIZE)
    true_c = torch.zeros(VOCAB_SIZE)
    amostra_p, amostra_y, n_auc = [], [], 0
    with torch.no_grad():
        for x, y in dl:
            logits = logits_de(modelo, x.to(device)).reshape(-1, VOCAB_SIZE)
            y = y.to(device).reshape(-1)
            m = y != PAD_ID
            logits, y = logits[m], y[m]
            if y.numel() == 0:
                continue
            perda += criterio_soma(logits, y).item()
            total += y.numel()
            pred = logits.argmax(-1)
            certo = pred == y
            acertos += certo.sum().item()
            top5 += (logits.topk(5, dim=-1).indices == y[:, None]).any(-1).sum().item()
            tp += torch.bincount(y[certo], minlength=VOCAB_SIZE).cpu().float()
            pred_c += torch.bincount(pred, minlength=VOCAB_SIZE).cpu().float()
            true_c += torch.bincount(y, minlength=VOCAB_SIZE).cpu().float()
            if calcular_auc and n_auc < max_auc:
                amostra_p.append(F.softmax(logits, dim=-1).cpu().numpy())
                amostra_y.append(y.cpu().numpy())
                n_auc += y.numel()
    if total == 0:
        raise ValueError('Conjunto de avaliação vazio.')
    presentes = true_c > 0
    precisao_c = tp / pred_c.clamp(min=1)
    recall_c = tp / true_c.clamp(min=1)
    f1_c = 2 * precisao_c * recall_c / (precisao_c + recall_c).clamp(min=1e-12)
    resultado = {
        'loss': perda / total,
        'perplexidade': math.exp(perda / total),
        'acuracia': acertos / total,
        'top5': top5 / total,
        'precisao_macro': precisao_c[presentes].mean().item(),
        'recall_macro': recall_c[presentes].mean().item(),
        'f1_macro': f1_c[presentes].mean().item(),
        'tokens_avaliados': total,
    }
    if calcular_auc:
        resultado['auc_roc_ovr'] = auc_ovr(np.concatenate(amostra_p)[:max_auc], np.concatenate(amostra_y)[:max_auc])
    return resultado

def salvar_checkpoint(nome, modelo, caminho):
    ckpt = {'state_dict': modelo.state_dict(), 'vocab_size': VOCAB_SIZE, 'vocabulario': vocab_json}
    if nome == 'transformer':
        ckpt.update({'d_model': D_MODEL, 'nhead': NHEAD, 'num_layers': NUM_LAYERS,
                     'dim_feedforward': DIM_FF, 'max_seq_len': MAX_SEQ_LEN})
    else:
        ckpt.update({'embed_dim': D_MODEL, 'hidden_dim': DIM_FF, 'num_layers': NUM_LAYERS})
    torch.save(ckpt, caminho)

def treinar_modelo(nome, epocas):
    modelo = criar_modelo(nome)
    dl_treino = DataLoader(JanelasDataset(dados_treino), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    dl_val = DataLoader(JanelasDataset(dados_val), batch_size=BATCH_SIZE, shuffle=False)
    criterio = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    otim = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(otim, T_max=epocas)
    caminho = PASTA_MODELOS / f'modelo_{nome}.pt'
    historico, melhor = [], float('inf')
    print(f'Treinando {nome.upper()} | parâmetros: {sum(p.numel() for p in modelo.parameters()):,}')
    for ep in range(1, epocas + 1):
        t0 = time.time()
        modelo.train()
        soma = 0.0
        for x, y in dl_treino:
            x, y = x.to(device), y.to(device)
            otim.zero_grad()
            loss = criterio(logits_de(modelo, x).reshape(-1, VOCAB_SIZE), y.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            otim.step()
            soma += loss.item()
        sched.step()
        val = avaliar(modelo, dl_val)
        registro = {
            'epoca': ep,
            'loss_treino': soma / len(dl_treino),
            'loss_val': val['loss'],
            'perplexidade_val': val['perplexidade'],
            'acuracia_val': val['acuracia'],
            'top5_val': val['top5'],
            'f1_macro_val': val['f1_macro'],
            'tempo_s': time.time() - t0,
        }
        historico.append(registro)
        print(f"Época {ep:02d}/{epocas} | treino {registro['loss_treino']:.4f} | val {val['loss']:.4f} | "
              f"ppl {val['perplexidade']:.2f} | acc {val['acuracia']:.4f} | top5 {val['top5']:.4f} | {registro['tempo_s']:.0f}s")
        if val['loss'] < melhor:
            melhor = val['loss']
            salvar_checkpoint(nome, modelo, caminho)
            print('  melhor modelo salvo no Drive')
    with open(PASTA_RELATORIOS / f'historico_{nome}.csv', 'w', newline='', encoding='utf-8') as f:
        escritor = csv.DictWriter(f, fieldnames=list(historico[0].keys()))
        escritor.writeheader()
        escritor.writerows(historico)
    return historico

def mascara_duracao():
    m = torch.zeros(VOCAB_SIZE, dtype=torch.bool)
    for t, i in token_para_id.items():
        if t.startswith('DURATION_'):
            m[i] = True
    return m.to(device)

def gerar_transformer(modelo, n_tokens=TOKENS_POR_MELODIA, temperatura=0.85, top_k=12, top_p=0.88, repeticao=1.35):
    modelo.eval()
    mask_dur = mascara_duracao()
    seq = [START_ID, token_para_id.get('NOTE_60', token_para_id.get('NOTE_64'))]
    tokens = [id_para_token[str(seq[1])]]
    ultimo = 'NOTE'
    with torch.no_grad():
        for _ in range(n_tokens):
            logits = logits_de(modelo, torch.tensor([seq[-MAX_SEQ_LEN:]], device=device))[0, -1].float().clone()
            for pid in set(seq[-8:]):
                logits[pid] = logits[pid] / repeticao if logits[pid] > 0 else logits[pid] * repeticao
            if ultimo == 'NOTE':
                logits[~mask_dur] = -float('inf')
                if ID_DUR_MIN is not None:
                    logits[ID_DUR_MIN] -= 2.0
            else:
                logits[mask_dur] = -float('inf')
            for especial in (PAD_ID, START_ID, END_ID):
                logits[especial] = -float('inf')
            logits = logits / max(temperatura, 0.01)
            if top_k > 0:
                limiar = torch.topk(logits, min(top_k, VOCAB_SIZE)).values[-1]
                logits[logits < limiar] = -float('inf')
            if 0.0 < top_p < 1.0:
                ordenados, idx = torch.sort(logits, descending=True)
                acumulada = torch.cumsum(F.softmax(ordenados, dim=-1), dim=-1)
                remover = acumulada > top_p
                remover[1:] = remover[:-1].clone()
                remover[0] = False
                logits[idx[remover]] = -float('inf')
            pid = torch.multinomial(F.softmax(logits, dim=-1), 1).item()
            tok = id_para_token[str(pid)]
            tokens.append(tok)
            seq.append(pid)
            ultimo = 'NOTE' if tok.startswith('NOTE_') else ('DURATION' if tok.startswith('DURATION_') else 'REST')
    return tokens

def gerar_lstm(modelo, n_tokens=TOKENS_POR_MELODIA, temperatura=0.9, top_k=10):
    modelo.eval()
    tokens = []
    with torch.no_grad():
        logits, hidden = modelo(torch.tensor([[START_ID]], device=device))
        for _ in range(n_tokens):
            last = logits[:, -1, :] / max(temperatura, 0.01)
            last[:, PAD_ID] = -float('inf')
            last[:, START_ID] = -float('inf')
            vals, idxs = torch.topk(last, min(top_k, VOCAB_SIZE))
            escolhido = idxs.gather(-1, torch.multinomial(F.softmax(vals, dim=-1), 1)).item()
            tok = id_para_token[str(escolhido)]
            if tok == '<END>':
                break
            tokens.append(tok)
            logits, hidden = modelo(torch.tensor([[escolhido]], device=device), hidden)
    return tokens

def stats_melodia(tokens):
    notas = [int(t[5:]) for t in tokens if t.startswith('NOTE_')]
    if len(notas) < 2:
        return None
    cont = Counter(notas)
    intervalos = [abs(b - a) for a, b in zip(notas, notas[1:])]
    return {
        'n_notas': len(notas),
        'alturas_distintas': len(cont),
        'participacao_nota_dominante': max(cont.values()) / len(notas),
        'amplitude_semitons': max(notas) - min(notas),
        'intervalo_medio_semitons': float(np.mean(intervalos)),
        'pct_repeticoes_consecutivas': sum(1 for i in intervalos if i == 0) / len(intervalos),
    }

def media_stats(lista):
    validos = [s for s in lista if s]
    return {k: float(np.mean([s[k] for s in validos])) for k in validos[0]} if validos else {}

def tokens_reais(pasta, n=500):
    arqs = sorted(p for p in pasta.glob('*.json') if p.stem != 'vocabulario')[:n]
    return [json.loads(a.read_text(encoding='utf-8'))['tokens'] for a in arqs]

def finalizar_modelo(nome, gerador):
    modelo = criar_modelo(nome)
    ckpt = torch.load(PASTA_MODELOS / f'modelo_{nome}.pt', map_location=device, weights_only=False)
    modelo.load_state_dict(ckpt['state_dict'])
    teste = avaliar(modelo, DataLoader(JanelasDataset(dados_teste), batch_size=BATCH_SIZE), calcular_auc=True)
    torch.manual_seed(42)
    melodias = [gerador(modelo) for _ in range(N_MELODIAS_GERADAS)]
    return {
        'teste': teste,
        'geracao': media_stats([stats_melodia(m) for m in melodias]),
        'real_teste': media_stats([stats_melodia(t) for t in tokens_reais(pastas['teste'])]),
        'exemplo_tokens': melodias[0][:24],
    }

def salvar_resultado(nome, historico, resultado):
    dados = {'modelo': nome, 'vocabulario_tamanho': VOCAB_SIZE, 'historico': historico, **resultado}
    (PASTA_RELATORIOS / f'resultados_{nome}.json').write_text(json.dumps(dados, ensure_ascii=False, indent=2), encoding='utf-8')
    t = resultado['teste']
    print(f"Teste {nome}: loss {t['loss']:.4f} | ppl {t['perplexidade']:.2f} | acc {t['acuracia']:.4f} | "
          f"top5 {t['top5']:.4f} | F1 macro {t['f1_macro']:.4f} | AUC {t['auc_roc_ovr']:.4f}")
    print('Estatísticas salvas em:', PASTA_RELATORIOS / f'resultados_{nome}.json')

print('Funções prontas.')

--- 
## 8. Treinar o Transformer (GPU T4)
Checkpoint em `modelos/modelo_transformer.pt`; histórico em `relatorios/historico_transformer.csv`; resultados em `relatorios/resultados_transformer.json`.

In [ ]:
historico_transformer = treinar_modelo('transformer', EPOCAS_TRANSFORMER)
resultado_transformer = finalizar_modelo('transformer', gerar_transformer)
salvar_resultado('transformer', historico_transformer, resultado_transformer)

--- 
## 9. Treinar a LSTM (GPU T4)
Checkpoint em `modelos/modelo_lstm.pt`; histórico em `relatorios/historico_lstm.csv`; resultados em `relatorios/resultados_lstm.json`.

In [ ]:
historico_lstm = treinar_modelo('lstm', EPOCAS_LSTM)
resultado_lstm = finalizar_modelo('lstm', gerar_lstm)
salvar_resultado('lstm', historico_lstm, resultado_lstm)

--- 
## 10. Estatísticas Comparativas
Gera o gráfico das curvas de treinamento e a tabela comparando os dois modelos, com as melodias reais do teste como referência.

In [ ]:
import pandas as pd

resultados = {}
for nome in ['transformer', 'lstm']:
    caminho = PASTA_RELATORIOS / f'resultados_{nome}.json'
    if caminho.exists():
        resultados[nome] = json.loads(caminho.read_text(encoding='utf-8'))

fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
for nome, r in resultados.items():
    ep = [h['epoca'] for h in r['historico']]
    eixos[0].plot(ep, [h['loss_treino'] for h in r['historico']], label=f'{nome} treino')
    eixos[0].plot(ep, [h['loss_val'] for h in r['historico']], '--', label=f'{nome} validação')
    eixos[1].plot(ep, [h['acuracia_val'] for h in r['historico']], label=nome)
eixos[0].set_title('Loss por época')
eixos[0].set_xlabel('época')
eixos[1].set_title('Acurácia (validação) por época')
eixos[1].set_xlabel('época')
for eixo in eixos:
    eixo.legend()
fig.tight_layout()
fig.savefig(PASTA_RELATORIOS / 'curvas_treinamento.png', dpi=120)
plt.show()

linhas = []
for nome, r in resultados.items():
    t = r['teste']
    linhas.append({'modelo': nome, 'loss_teste': t['loss'], 'perplexidade': t['perplexidade'],
                   'acuracia': t['acuracia'], 'top5': t['top5'], 'precisao_macro': t['precisao_macro'],
                   'recall_macro': t['recall_macro'], 'f1_macro': t['f1_macro'], 'auc_roc_ovr': t['auc_roc_ovr']})
tabela_teste = pd.DataFrame(linhas).set_index('modelo')
print('Desempenho no conjunto de teste')
display(tabela_teste.round(4))

linhas_g = []
for nome, r in resultados.items():
    linhas_g.append({'fonte': nome, **r['geracao']})
linhas_g.append({'fonte': 'real (teste)', **next(iter(resultados.values()))['real_teste']})
tabela_geracao = pd.DataFrame(linhas_g).set_index('fonte')
print('Estatísticas das melodias (geradas x reais)')
display(tabela_geracao.round(3))

(PASTA_RELATORIOS / 'tabela_comparativa.csv').write_text(
    tabela_teste.round(6).to_csv() + '\n' + tabela_geracao.round(6).to_csv(), encoding='utf-8')
print('Gráfico e tabelas salvos em:', PASTA_RELATORIOS)

--- 
## 11. Conferência dos Arquivos Persistidos no Drive

In [ ]:
for pasta in [PASTA_DADOS, PASTA_MODELOS, PASTA_RELATORIOS]:
    arquivos = [a for a in pasta.rglob('*') if a.is_file()]
    print(f'{pasta}: {len(arquivos)} arquivos')
print('Checkpoints:', [p.name for p in PASTA_MODELOS.glob('*.pt')])